# Lab 04-04: Foundation Model APIs

**Module:** 04 — Assembling and Deploying Applications  
**Exam Weight:** ~10 questions across Module 04 (22% of exam)  
**UI Sections:** Serving | Playground | Discover  
**Difficulty:** Intermediate  
**Estimated Time:** 40–50 minutes

---

## Learning Objectives

- Understand Foundation Model APIs as **pre-deployed, pay-per-token** endpoints hosted by Databricks
- Compare the three model serving options: **Foundation Model APIs**, **Provisioned Throughput**, and **External Models (AI Gateway)**
- Call Foundation Model APIs using the **OpenAI-compatible SDK** for chat completions and embeddings
- Estimate token usage and costs for Foundation Model API calls
- Know when to upgrade from pay-per-token to provisioned throughput

---

## Exam Objectives Covered

- **Identify the appropriate model serving option for a given use case**
- **Call Foundation Model APIs using the OpenAI-compatible interface**
- **Distinguish between Foundation Model APIs, provisioned throughput, and external models**

---

## What Are Foundation Model APIs and Why Do They Matter?

Foundation Model APIs are **pre-deployed, pay-per-token endpoints** hosted by Databricks. They provide instant access to popular open-source models — no deployment, no infrastructure, no waiting. You call an API, you get tokens back, you pay for what you use.

This is the **fastest path from zero to working LLM application** on Databricks. There is no model to deploy, no endpoint to configure, no GPU to provision. The models are already running and waiting for your API call.

The exam specifically tests whether you can:
1. **Identify** which serving option to choose for a given scenario
2. **Call** Foundation Model APIs using the OpenAI-compatible interface
3. **Distinguish** between the three serving options and when each applies

| Serving Option | What It Is | When to Use It |
|----------------|-----------|----------------|
| **Foundation Model APIs** | Pre-deployed, pay-per-token | Prototyping, variable workloads, getting started |
| **Provisioned Throughput** | Reserved GPU capacity, fixed cost | Production with SLAs, HIPAA, fine-tuned models |
| **External Models (AI Gateway)** | Proxy to OpenAI, Anthropic, etc. | Need GPT-4, Claude, or other non-Databricks models |

---

## Mental Model

> **Analogy:** Foundation Model APIs are like a utility — turn on the tap (API call) and water (tokens) flows. You pay by the gallon (token). Provisioned throughput is like buying your own water tank — fixed cost, guaranteed supply, no matter how much you use. External models (AI Gateway) are like importing bottled water — same tap interface, different source (OpenAI, Anthropic, etc.).

The key insight is that **all three use the same OpenAI-compatible API interface**. Your code looks identical whether you are calling a Foundation Model API, a provisioned throughput endpoint, or an external model. The difference is in billing, performance guarantees, and which models are available.

---

## Exam Alert: Common Traps

The exam loves to test whether you understand the **differences** between serving options. Watch for these:

| # | Trap Statement | Correct Answer |
|---|---------------|----------------|
| 1 | "Foundation Model APIs require you to deploy a model" | **Wrong.** They are pre-deployed by Databricks — just call the endpoint. No deployment step needed. |
| 2 | "All Foundation Models have the same rate limits" | **Wrong.** Rate limits vary by model and workspace tier. Larger models may have lower rate limits. |
| 3 | "Provisioned throughput and pay-per-token have the same API" | **Partially true, but misleading.** Same API *interface* (OpenAI-compatible), but different billing, latency guarantees, and available models. |
| 4 | "You need provisioned throughput to use Foundation Model APIs" | **Wrong.** Foundation Model APIs are the pay-per-token tier — provisioned throughput is a separate, higher tier. |
| 5 | "External models are only for testing" | **Wrong.** External models (AI Gateway) are production-ready — they let you use OpenAI, Anthropic, etc. through a governed Databricks endpoint. |

---

## Prerequisites and UI Navigation

### What You Need
- A Databricks workspace with Unity Catalog enabled
- Access to a cluster with the Databricks Runtime ML
- The `databricks-sdk` and `openai` packages installed

### Key UI Paths
- **UI →** Left nav → **Serving** → View all serving endpoints
- **UI →** Left nav → **Playground** → Test Foundation Models interactively
- **UI →** Left nav → **Marketplace** → **Discover** → Browse available models

---

## Setup

In [ ]:
# Install required packages
%pip install openai databricks-sdk -q
dbutils.library.restartPython()

In [ ]:
# ---- Imports and Configuration ----
import os
import json
from openai import OpenAI

# Databricks automatically provides these environment variables on clusters
DATABRICKS_HOST = os.environ.get(
    "DATABRICKS_HOST",
    "https://" + spark.conf.get("spark.databricks.workspaceUrl")
)
DATABRICKS_TOKEN = os.environ.get(
    "DATABRICKS_TOKEN",
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
)

# The OpenAI client works with Databricks Foundation Model APIs
# because Databricks exposes an OpenAI-compatible REST interface
client = OpenAI(
    api_key=DATABRICKS_TOKEN,
    base_url=f"{DATABRICKS_HOST}/serving-endpoints"
)

# Configuration
MODEL_CHAT = "databricks-meta-llama-3-3-70b-instruct"  # Chat completions
MODEL_EMBED = "databricks-gte-large-en"                 # Embeddings
CATALOG = "workspace"
SCHEMA = "genai_labs"

print(f"OpenAI client configured for: {DATABRICKS_HOST}")
print(f"Chat model:      {MODEL_CHAT}")
print(f"Embedding model: {MODEL_EMBED}")

### Expected Output

```
OpenAI client configured for: https://<your-workspace>.cloud.databricks.com
Chat model:      databricks-meta-llama-3-3-70b-instruct
Embedding model: databricks-gte-large-en
```

### What Just Happened

We created an **OpenAI client** pointed at Databricks. This is the core pattern — Databricks Foundation Model APIs are **OpenAI-compatible**, meaning you use the exact same SDK and code patterns you would use with OpenAI's API. The only difference is the `base_url` pointing to your Databricks workspace.

---

## Step 1: Pay-per-Token vs Provisioned Throughput — The Two Tiers

Foundation Model APIs come in two tiers. Understanding when to use each is an exam-critical skill.

| Feature | Pay-per-Token | Provisioned Throughput |
|---------|--------------|----------------------|
| **Billing** | Per-token (like a taxi meter) | Fixed rate per hour (like a rental car) |
| **Setup** | Zero — models are pre-deployed | Create a serving endpoint with reserved capacity |
| **Latency** | Variable (shared infrastructure) | Predictable (dedicated GPUs) |
| **Rate Limits** | Workspace-level limits | Your reserved capacity, no sharing |
| **SLA** | Best effort | Guaranteed throughput |
| **Fine-tuned Models** | Not supported | Supported |
| **HIPAA Compliance** | Not guaranteed | Available |
| **Best For** | Prototyping, dev, variable traffic | Production, predictable traffic, compliance |

### Decision Rule for the Exam

If the question mentions any of these words, the answer is **provisioned throughput**:
- "production", "SLA", "HIPAA", "fine-tuned", "guaranteed", "predictable latency"

If the question mentions any of these words, the answer is **pay-per-token** (Foundation Model APIs):
- "prototype", "experiment", "variable traffic", "low cost", "getting started"

---

## Step 2: Calling Foundation Model APIs — Chat Completions

The most common use case is **chat completions** — sending a conversation to the model and getting a response. The API is identical to OpenAI's.

In [ ]:
# ---- Step 2a: Basic chat completion ----
# This is the most fundamental Foundation Model API call.
# Notice: the code is IDENTICAL to what you would write for OpenAI.

response = client.chat.completions.create(
    model=MODEL_CHAT,
    messages=[
        {"role": "system", "content": "You are a helpful assistant that explains concepts concisely."},
        {"role": "user", "content": "What is a Foundation Model API on Databricks? Answer in 2-3 sentences."}
    ],
    max_tokens=200,
    temperature=0.1  # Low temperature for factual, consistent responses
)

# Extract the response
answer = response.choices[0].message.content
usage = response.usage

print("=== Chat Completion Response ===")
print(f"Model: {response.model}")
print(f"Answer: {answer}")
print()
print("=== Token Usage ===")
print(f"Prompt tokens:     {usage.prompt_tokens}")
print(f"Completion tokens: {usage.completion_tokens}")
print(f"Total tokens:      {usage.total_tokens}")

### Expected Output

```
=== Chat Completion Response ===
Model: databricks-meta-llama-3-3-70b-instruct
Answer: Foundation Model APIs on Databricks are pre-deployed, pay-per-token endpoints
that provide instant access to popular open-source LLMs. They use an OpenAI-compatible
interface, so you can call them using the standard OpenAI SDK without deploying or
managing any infrastructure.

=== Token Usage ===
Prompt tokens:     38
Completion tokens: 52
Total tokens:      90
```

### What Just Happened

We made a **chat completion** call using the OpenAI SDK. The code is identical to calling OpenAI directly — the only difference is the `base_url` pointing to Databricks. This is the key exam concept: **Foundation Model APIs are OpenAI-compatible**.

In [ ]:
# ---- Step 2b: Chat completion with structured JSON output ----
# You can request JSON output for structured responses.
# This is useful for agents and tool-calling patterns.

response_json = client.chat.completions.create(
    model=MODEL_CHAT,
    messages=[
        {"role": "system", "content": "You are a data classifier. Respond ONLY with valid JSON."},
        {"role": "user", "content": """Classify this customer message:
\"I've been waiting 3 weeks for my order and nobody has responded to my emails!\"

Return JSON with keys: sentiment, urgency, category, suggested_action"""}
    ],
    max_tokens=200,
    temperature=0.0,
    response_format={"type": "json_object"}  # Force JSON output
)

result = json.loads(response_json.choices[0].message.content)
print("=== Structured Classification ===")
print(json.dumps(result, indent=2))
print()
print(f"Tokens used: {response_json.usage.total_tokens}")

### Expected Output

```
=== Structured Classification ===
{
  "sentiment": "negative",
  "urgency": "high",
  "category": "order_delay",
  "suggested_action": "escalate_to_support_lead"
}

Tokens used: 128
```

### What Just Happened

We used `response_format={"type": "json_object"}` to force the model to return valid JSON. This is critical for **agentic workflows** where downstream code needs to parse the model's output programmatically.

---

## Step 3: Calling Foundation Model APIs — Embeddings

Embedding endpoints convert text into dense vectors. These are essential for **RAG (Retrieval-Augmented Generation)** — you embed documents, store them in Vector Search, then embed queries to find relevant documents.

In [ ]:
# ---- Step 3: Generate embeddings via Foundation Model API ----
# Embeddings convert text into numerical vectors for semantic search.

texts_to_embed = [
    "Foundation Model APIs are pay-per-token endpoints on Databricks.",
    "Provisioned throughput provides reserved GPU capacity.",
    "External models connect to OpenAI and Anthropic through AI Gateway.",
    "Vector Search uses embeddings to find similar documents."
]

response_embed = client.embeddings.create(
    model=MODEL_EMBED,
    input=texts_to_embed
)

# Show results
print(f"=== Embedding Results ===")
print(f"Model: {MODEL_EMBED}")
print(f"Number of texts embedded: {len(response_embed.data)}")
print(f"Embedding dimension: {len(response_embed.data[0].embedding)}")
print(f"Total tokens used: {response_embed.usage.total_tokens}")
print()
for i, item in enumerate(response_embed.data):
    vec = item.embedding
    print(f"Text {i+1}: [{vec[0]:.6f}, {vec[1]:.6f}, {vec[2]:.6f}, ... ] (length={len(vec)})")

### Expected Output

```
=== Embedding Results ===
Model: databricks-gte-large-en
Number of texts embedded: 4
Embedding dimension: 1024
Total tokens used: 52

Text 1: [0.012345, -0.067890, 0.034567, ... ] (length=1024)
Text 2: [0.023456, -0.045678, 0.056789, ... ] (length=1024)
Text 3: [-0.034567, 0.012345, 0.078901, ... ] (length=1024)
Text 4: [0.045678, -0.023456, 0.012345, ... ] (length=1024)
```

### What Just Happened

We called the **embedding endpoint** to convert 4 text strings into 1024-dimensional vectors. These vectors capture the *semantic meaning* of each text. In a RAG pipeline, you would store these in **Databricks Vector Search** and use cosine similarity to find relevant documents at query time.

---

## Step 4: Token Counting and Cost Estimation

Understanding token usage is essential for cost management. Foundation Model APIs charge per token, so knowing how to estimate costs helps you budget and optimize.

In [ ]:
# ---- Step 4a: Token counting and cost estimation ----
# Foundation Model APIs charge per token. Let's estimate costs.

# Approximate pricing (check Databricks docs for current rates)
# These are illustrative — actual pricing varies by model and region
PRICE_PER_1K_INPUT_TOKENS = 0.001    # $0.001 per 1K input tokens (example)
PRICE_PER_1K_OUTPUT_TOKENS = 0.002   # $0.002 per 1K output tokens (example)

def estimate_cost(prompt_tokens, completion_tokens):
    """Estimate cost for a Foundation Model API call."""
    input_cost = (prompt_tokens / 1000) * PRICE_PER_1K_INPUT_TOKENS
    output_cost = (completion_tokens / 1000) * PRICE_PER_1K_OUTPUT_TOKENS
    return input_cost + output_cost

# Simulate a batch of API calls
scenarios = [
    {"name": "Single question",        "input_tokens": 50,    "output_tokens": 100},
    {"name": "RAG with context",        "input_tokens": 2000,  "output_tokens": 500},
    {"name": "Document summarization",  "input_tokens": 10000, "output_tokens": 1000},
    {"name": "1000 batch classif.",     "input_tokens": 100000, "output_tokens": 50000},
]

print("=== Cost Estimation (Foundation Model APIs — Pay-per-Token) ===")
print(f"{'Scenario':<28} {'Input Tokens':>14} {'Output Tokens':>14} {'Est. Cost':>12}")
print("-" * 72)
total_cost = 0
for s in scenarios:
    cost = estimate_cost(s["input_tokens"], s["output_tokens"])
    total_cost += cost
    print(f"{s['name']:<28} {s['input_tokens']:>14,} {s['output_tokens']:>14,} {f'${cost:.4f}':>12}")
print("-" * 72)
print(f"{'TOTAL':<28} {'':>14} {'':>14} {f'${total_cost:.4f}':>12}")

### Expected Output

```
=== Cost Estimation (Foundation Model APIs — Pay-per-Token) ===
Scenario                       Input Tokens  Output Tokens     Est. Cost
------------------------------------------------------------------------
Single question                          50            100       $0.0003
RAG with context                      2,000            500       $0.0030
Document summarization               10,000          1,000       $0.0120
1000 batch classif.                 100,000         50,000       $0.2000
------------------------------------------------------------------------
TOTAL                                                             $0.2153
```

### What Just Happened

We estimated costs for different usage patterns. Key takeaway: **pay-per-token is extremely cost-effective for prototyping and variable workloads**, but costs can add up for high-volume batch processing — that is when **provisioned throughput** becomes more economical.

In [ ]:
# ---- Step 4b: Break-even analysis — when does provisioned throughput save money? ----
# Provisioned throughput charges a fixed hourly rate regardless of usage.
# At some volume, it becomes cheaper than pay-per-token.

PROVISIONED_HOURLY_RATE = 5.00  # Example: $5/hour for provisioned throughput

def monthly_pay_per_token(daily_tokens):
    """Estimate monthly cost at pay-per-token rates."""
    daily_input = daily_tokens * 0.6   # Assume 60% input
    daily_output = daily_tokens * 0.4  # Assume 40% output
    daily_cost = estimate_cost(int(daily_input), int(daily_output))
    return daily_cost * 30

def monthly_provisioned(hours_per_day=24):
    """Estimate monthly cost for provisioned throughput."""
    return PROVISIONED_HOURLY_RATE * hours_per_day * 30

print("=== Break-Even Analysis: Pay-per-Token vs Provisioned Throughput ===")
print(f"{'Daily Tokens':>15} {'Pay-per-Token/mo':>20} {'Provisioned/mo':>18} {'Winner':>15}")
print("-" * 72)
for daily in [10_000, 100_000, 500_000, 1_000_000, 5_000_000, 10_000_000]:
    ppt_cost = monthly_pay_per_token(daily)
    prov_cost = monthly_provisioned()
    winner = "Pay-per-token" if ppt_cost < prov_cost else "Provisioned"
    print(f"{daily:>15,} {f'${ppt_cost:,.2f}':>20} {f'${prov_cost:,.2f}':>18} {winner:>15}")

### Expected Output

```
=== Break-Even Analysis: Pay-per-Token vs Provisioned Throughput ===
   Daily Tokens   Pay-per-Token/mo   Provisioned/mo          Winner
------------------------------------------------------------------------
         10,000               $0.42          $3,600.00   Pay-per-token
        100,000               $4.20          $3,600.00   Pay-per-token
        500,000              $21.00          $3,600.00   Pay-per-token
      1,000,000              $42.00          $3,600.00   Pay-per-token
      5,000,000             $210.00          $3,600.00   Pay-per-token
     10,000,000             $420.00          $3,600.00   Pay-per-token
```

### What Just Happened

This analysis shows that **provisioned throughput is about dedicated capacity and SLAs, not just cost**. The real reasons to choose provisioned throughput are:
- **Guaranteed latency** (no noisy-neighbor effects)
- **HIPAA compliance** requirements
- **Fine-tuned model** deployment (only available on provisioned)
- **Production SLAs** that require predictable performance

> **Exam Tip:** The exam will NOT ask you to calculate costs. It will ask you to **choose the right tier for a scenario**. Focus on the qualitative differences, not the math.

---

## Step 5: Provisioned Throughput — When and Why

Provisioned throughput is the **production tier** of Foundation Model APIs. Instead of sharing infrastructure with other users (pay-per-token), you get dedicated GPU capacity reserved just for your workload.

### When to Use Provisioned Throughput

| Scenario | Why Provisioned? |
|----------|------------------|
| Customer-facing chatbot with SLA | Guaranteed response times, no shared capacity |
| Healthcare application (HIPAA) | Provisioned throughput offers HIPAA compliance |
| Serving a fine-tuned model | Fine-tuned models can ONLY be served via provisioned throughput |
| High-volume batch inference | Predictable costs at scale |
| Latency-sensitive application | Dedicated GPUs = consistent latency |

In [ ]:
# ---- Step 5: Creating a provisioned throughput endpoint (conceptual) ----
# NOTE: Actually creating a provisioned endpoint requires admin permissions
# and incurs real costs. This cell shows the API pattern.

# from databricks.sdk import WorkspaceClient
# w = WorkspaceClient()
#
# # Step 1: Create the endpoint with reserved capacity
# endpoint = w.serving_endpoints.create(
#     name="my-production-llama",
#     config={
#         "served_entities": [{
#             "entity_name": "databricks-meta-llama-3-3-70b-instruct",
#             "min_provisioned_throughput": 1000,  # Minimum tokens/second
#             "max_provisioned_throughput": 5000,  # Maximum tokens/second
#         }]
#     }
# )
#
# # Step 2: Call it (same OpenAI-compatible interface!)
# client_prov = OpenAI(
#     api_key=DATABRICKS_TOKEN,
#     base_url=f"{DATABRICKS_HOST}/serving-endpoints"
# )
# response = client_prov.chat.completions.create(
#     model="my-production-llama",  # Your custom endpoint name
#     messages=[{"role": "user", "content": "Hello!"}]
# )

# UI Walkthrough for creating provisioned throughput
print("=== Creating Provisioned Throughput (UI Walkthrough) ===")
print()
print("**UI ->** Left nav -> **Serving** -> **Create serving endpoint**")
print()
print("Step 1: Name your endpoint (e.g., 'my-production-llama')")
print("Step 2: Select 'Foundation Model API' as the entity type")
print("Step 3: Choose your model (e.g., Meta-Llama-3.3-70B-Instruct)")
print("Step 4: Set 'Provisioned Throughput' (not pay-per-token)")
print("Step 5: Configure min/max tokens per second")
print("Step 6: Click 'Create' -- endpoint provisions in ~10-15 minutes")
print()
print("KEY POINT: Once created, you call it with the SAME OpenAI SDK --")
print("just change the model name to your endpoint name.")

### Expected Output

```
=== Creating Provisioned Throughput (UI Walkthrough) ===

**UI ->** Left nav -> **Serving** -> **Create serving endpoint**

Step 1: Name your endpoint (e.g., 'my-production-llama')
Step 2: Select 'Foundation Model API' as the entity type
Step 3: Choose your model (e.g., Meta-Llama-3.3-70B-Instruct)
Step 4: Set 'Provisioned Throughput' (not pay-per-token)
Step 5: Configure min/max tokens per second
Step 6: Click 'Create' -- endpoint provisions in ~10-15 minutes

KEY POINT: Once created, you call it with the SAME OpenAI SDK --
just change the model name to your endpoint name.
```

### What Just Happened

We walked through the **UI and API patterns** for provisioned throughput. The critical exam insight: **the API interface is identical** — you just change the endpoint name. The difference is behind the scenes (dedicated GPUs, SLAs, billing).

---

## Step 6: External Models via AI Gateway

External models let you call **non-Databricks models** (OpenAI GPT-4, Anthropic Claude, Cohere, etc.) through a Databricks-managed endpoint. This is the **AI Gateway** pattern — a single, governed interface for all your LLM providers.

### Why Use AI Gateway Instead of Calling Providers Directly?

| Benefit | Description |
|---------|-------------|
| **Unified Interface** | Same OpenAI-compatible SDK for all providers |
| **Governance** | All calls logged in Unity Catalog, access controlled by ACLs |
| **Rate Limiting** | Databricks manages rate limits across your organization |
| **Cost Tracking** | All usage tracked in one place |
| **Secret Management** | API keys stored securely, not in notebooks |
| **Fallback Routing** | Route to backup providers if primary is down |

In [ ]:
# ---- Step 6: External Models -- AI Gateway Configuration (conceptual) ----
# External model endpoints proxy calls to third-party LLM providers.
# This is done through the Databricks UI or SDK.

# from databricks.sdk import WorkspaceClient
# from databricks.sdk.service.serving import (
#     EndpointCoreConfigInput,
#     ServedEntityInput,
#     ExternalModel,
#     OpenAiConfig,
# )
#
# w = WorkspaceClient()
#
# # Create an external model endpoint for OpenAI GPT-4
# w.serving_endpoints.create(
#     name="openai-gpt4-gateway",
#     config=EndpointCoreConfigInput(
#         served_entities=[ServedEntityInput(
#             external_model=ExternalModel(
#                 name="gpt-4",
#                 provider="openai",
#                 task="llm/v1/chat",
#                 openai_config=OpenAiConfig(
#                     openai_api_key="{{secrets/my-scope/openai-key}}"
#                 )
#             )
#         )]
#     )
# )
#
# # Once created, call it with the SAME OpenAI SDK:
# response = client.chat.completions.create(
#     model="openai-gpt4-gateway",  # Your external endpoint name
#     messages=[{"role": "user", "content": "Hello from AI Gateway!"}]
# )

# UI Walkthrough
print("=== External Models -- AI Gateway (UI Walkthrough) ===")
print()
print("**UI ->** Left nav -> **Serving** -> **Create serving endpoint**")
print()
print("Step 1: Name your endpoint (e.g., 'openai-gpt4-gateway')")
print("Step 2: Select 'External model' as the entity type")
print("Step 3: Choose provider: OpenAI, Anthropic, Cohere, AI21, etc.")
print("Step 4: Select model: gpt-4, claude-3, command-r, etc.")
print("Step 5: Configure API key (use Databricks Secrets for security)")
print("Step 6: Set task type: Chat, Completions, or Embeddings")
print("Step 7: Click 'Create'")
print()
print("RESULT: You now call GPT-4 (or Claude, etc.) using the SAME")
print("        OpenAI SDK pointed at your Databricks workspace.")
print("        All calls are logged, governed, and rate-limited.")

### Expected Output

```
=== External Models -- AI Gateway (UI Walkthrough) ===

**UI ->** Left nav -> **Serving** -> **Create serving endpoint**

Step 1: Name your endpoint (e.g., 'openai-gpt4-gateway')
Step 2: Select 'External model' as the entity type
Step 3: Choose provider: OpenAI, Anthropic, Cohere, AI21, etc.
Step 4: Select model: gpt-4, claude-3, command-r, etc.
Step 5: Configure API key (use Databricks Secrets for security)
Step 6: Set task type: Chat, Completions, or Embeddings
Step 7: Click 'Create'

RESULT: You now call GPT-4 (or Claude, etc.) using the SAME
        OpenAI SDK pointed at your Databricks workspace.
        All calls are logged, governed, and rate-limited.
```

### What Just Happened

We walked through creating an **external model endpoint** via AI Gateway. This is how you bring non-Databricks models (GPT-4, Claude, etc.) into your Databricks ecosystem while maintaining governance, logging, and unified access control.

> **Exam Tip:** The exam may ask "How do you call OpenAI models from Databricks?" The answer is **AI Gateway (external models)** — not by putting your OpenAI API key directly in a notebook.

---

## Step 7: Listing Available Models

You can programmatically discover which Foundation Models are available in your workspace.

In [ ]:
# ---- Step 7: List available Foundation Model API endpoints ----
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# List all serving endpoints
endpoints = list(w.serving_endpoints.list())

print(f"=== Serving Endpoints in Your Workspace ({len(endpoints)} total) ===")
print()
print(f"{'Endpoint Name':<45} {'State':<12} {'Type'}")
print("-" * 80)

foundation_models = []
external_models = []
custom_models = []

for ep in sorted(endpoints, key=lambda x: x.name):
    state = ep.state.ready if ep.state else "UNKNOWN"
    
    # Categorize the endpoint
    if ep.name.startswith("databricks-"):
        ep_type = "Foundation Model API"
        foundation_models.append(ep.name)
    elif hasattr(ep, 'config') and ep.config and any(
        hasattr(se, 'external_model') and se.external_model
        for se in (ep.config.served_entities or [])
    ):
        ep_type = "External Model"
        external_models.append(ep.name)
    else:
        ep_type = "Custom/Provisioned"
        custom_models.append(ep.name)
    
    print(f"{ep.name:<45} {str(state):<12} {ep_type}")

print()
print(f"Summary: {len(foundation_models)} Foundation, "
      f"{len(external_models)} External, {len(custom_models)} Custom/Provisioned")

### Expected Output

```
=== Serving Endpoints in Your Workspace (12 total) ===

Endpoint Name                                 State        Type
--------------------------------------------------------------------------------
databricks-dbrx-instruct                      READY        Foundation Model API
databricks-gte-large-en                       READY        Foundation Model API
databricks-meta-llama-3-1-405b-instruct       READY        Foundation Model API
databricks-meta-llama-3-3-70b-instruct        READY        Foundation Model API
databricks-mixtral-8x7b-instruct              READY        Foundation Model API
...

Summary: 8 Foundation, 1 External, 3 Custom/Provisioned
```

### What Just Happened

We used the **Databricks SDK** to list all serving endpoints. Notice that Foundation Model APIs all start with `databricks-` — they are pre-deployed and always `READY`. Custom and provisioned endpoints are ones you or your team created.

---

## Step 8: Practice — Match Scenarios to Serving Options

This is exactly the kind of question you will see on the exam. For each scenario, identify the best serving option.

In [ ]:
# ---- Step 8: Exam-style practice -- match scenarios to serving options ----

scenarios = [
    {
        "scenario": "A data scientist wants to quickly test if an LLM can classify support tickets.",
        "answer": "Foundation Model APIs (pay-per-token)",
        "why": "Prototyping, variable traffic, no setup needed"
    },
    {
        "scenario": "A healthcare company needs HIPAA-compliant text generation for patient summaries.",
        "answer": "Provisioned Throughput",
        "why": "HIPAA compliance is only available with provisioned throughput"
    },
    {
        "scenario": "A team wants to use GPT-4 for complex reasoning but needs Databricks governance.",
        "answer": "External Models (AI Gateway)",
        "why": "GPT-4 is not a Databricks model -- use AI Gateway to proxy calls with governance"
    },
    {
        "scenario": "A production chatbot serves 10K customers/day and needs <2s response time SLA.",
        "answer": "Provisioned Throughput",
        "why": "Production SLA + predictable latency = provisioned throughput"
    },
    {
        "scenario": "A team wants to compare Llama 3.3, Mixtral, and DBRX on the same eval set.",
        "answer": "Foundation Model APIs (pay-per-token)",
        "why": "Model comparison/experimentation -- pay-per-token is cheapest and fastest to set up"
    },
    {
        "scenario": "A company fine-tuned Llama 3.3 on proprietary data and needs to serve it.",
        "answer": "Provisioned Throughput",
        "why": "Fine-tuned models can ONLY be served via provisioned throughput"
    },
]

print("=== Exam Practice: Match Scenario to Serving Option ===")
print()
for i, s in enumerate(scenarios, 1):
    print(f"Scenario {i}: {s['scenario']}")
    print(f"  -> Best option:  {s['answer']}")
    print(f"  -> Why:          {s['why']}")
    print()

### Expected Output

```
=== Exam Practice: Match Scenario to Serving Option ===

Scenario 1: A data scientist wants to quickly test if an LLM can classify support tickets.
  -> Best option:  Foundation Model APIs (pay-per-token)
  -> Why:          Prototyping, variable traffic, no setup needed

Scenario 2: A healthcare company needs HIPAA-compliant text generation for patient summaries.
  -> Best option:  Provisioned Throughput
  -> Why:          HIPAA compliance is only available with provisioned throughput

Scenario 3: A team wants to use GPT-4 for complex reasoning but needs Databricks governance.
  -> Best option:  External Models (AI Gateway)
  -> Why:          GPT-4 is not a Databricks model -- use AI Gateway to proxy calls with governance

Scenario 4: A production chatbot serves 10K customers/day and needs <2s response time SLA.
  -> Best option:  Provisioned Throughput
  -> Why:          Production SLA + predictable latency = provisioned throughput

Scenario 5: A team wants to compare Llama 3.3, Mixtral, and DBRX on the same eval set.
  -> Best option:  Foundation Model APIs (pay-per-token)
  -> Why:          Model comparison/experimentation -- pay-per-token is cheapest and fastest to set up

Scenario 6: A company fine-tuned Llama 3.3 on proprietary data and needs to serve it.
  -> Best option:  Provisioned Throughput
  -> Why:          Fine-tuned models can ONLY be served via provisioned throughput
```

### What Just Happened

We practiced the **exam-critical skill** of matching business scenarios to serving options. Remember the decision rules:
- **Prototype / experiment / variable** = Pay-per-token
- **Production / SLA / HIPAA / fine-tuned** = Provisioned Throughput
- **Non-Databricks model (GPT-4, Claude)** = External Models (AI Gateway)

---

## Exam Tips and Traps

| # | Tip or Trap | Explanation |
|---|-------------|-------------|
| 1 | **Trap:** "You must deploy a model before calling Foundation Model APIs" | Foundation Model APIs are **pre-deployed**. No deployment step. Just call the endpoint. |
| 2 | **Tip:** All three serving options use the same OpenAI-compatible interface | The SDK code is identical — only the endpoint name and billing change. |
| 3 | **Trap:** "Fine-tuned models work with pay-per-token" | Fine-tuned models require **provisioned throughput**. This is a hard rule. |
| 4 | **Tip:** AI Gateway = External Models = same concept | The exam may use either term. Both refer to proxying calls to third-party LLM providers. |
| 5 | **Trap:** "Put your OpenAI API key in a notebook to call GPT-4" | Use **AI Gateway** (external models) + Databricks Secrets. Never hardcode API keys. |
| 6 | **Tip:** Foundation Model API names start with `databricks-` | e.g., `databricks-meta-llama-3-3-70b-instruct`, `databricks-gte-large-en` |

---

## Key Takeaways

### Core Concepts

- **Foundation Model APIs** are pre-deployed, pay-per-token endpoints — zero setup, instant access
- **Provisioned Throughput** is reserved GPU capacity — for production, SLAs, HIPAA, and fine-tuned models
- **External Models (AI Gateway)** proxy calls to third-party providers (OpenAI, Anthropic, etc.) with governance
- All three use the **same OpenAI-compatible SDK** — your code does not change between tiers

### Architecture Summary

```
Your Code (OpenAI SDK)
    |
    v
Databricks Serving Endpoint
    |
    +---> Foundation Model API (pay-per-token, pre-deployed)
    |
    +---> Provisioned Throughput (reserved GPUs, your endpoint)
    |
    +---> External Model / AI Gateway (proxy to OpenAI, Anthropic, etc.)
```

### Exam Decision Tree

```
Need an LLM on Databricks?
    |
    +---> Is it a non-Databricks model (GPT-4, Claude)?
    |         YES -> External Models (AI Gateway)
    |
    +---> Is it for production with SLA/HIPAA/fine-tuned?
    |         YES -> Provisioned Throughput
    |
    +---> Otherwise (prototyping, experiments, variable traffic)
              -> Foundation Model APIs (pay-per-token)
```

---

## Cleanup

This lab did not create any persistent tables or endpoints. The only resources used were pay-per-token API calls, which have no ongoing cost.

---

## Next Lab

Continue to **Lab 04-04b: Persistent Memory and State** (`04b_persistent_memory.ipynb`), where you will implement five distinct memory patterns (buffer, summary, structured, feature tables, and checkpointing) using Delta tables and Unity Catalog.